# FlowDebug — of the records behind a result, which ones mattered

Every record of a group contributes to its aggregate, so provenance for a wrong `MAX`
returns the whole group. Influence asks *how much* each contributed, reading the
aggregate's own semantics.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("flowdebug-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## Influence over a MAX

Only the largest record influences a maximum. Provenance would return all four.

In [ ]:
ranked = bigasterisk.influence(spark).influencers(
    "SELECT cid, MAX(amount) AS peak FROM orders GROUP BY cid",
    faulty_where="peak > 1000")

for r in ranked:
    print(r)

## Influence over a SUM

Here influence is the size of the contribution, so the shares add to one.

In [ ]:
sums = bigasterisk.influence(spark).influencers(
    "SELECT cid, SUM(amount) AS total FROM orders GROUP BY cid",
    faulty_where="total > 50000")

for r in sums:
    print(r)

## Check

In [ ]:
assert abs(ranked[0].score - 1.0) < 1e-9 and ranked[0].row["amount"] == 99999
assert all(r.score == 0.0 for r in ranked[1:])
assert sums[0].score > 0.99
assert abs(sum(r.score for r in sums) - 1.0) < 1e-6
print("OK")